### Initial test in jupyter notebook

In [1]:
# The notebook will need couple module files from code directory.
import os,sys
# os.path.join('..', 'code') is the relative path for code folder, module_path convert it to absolute path
module_path = os.path.abspath(os.path.join('..', 'code'))

# append the code folder in sys.path if not exists
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
# documents: list of 1208 json object, 24 secs to load

# build_index is using minsearch.Index to build text index probably via TD_IDF
index = build_index(documents)
# index: minsearch.Index object
openai_client = OpenAI()

# assistant = RAGBase(
#     index=index,
#     llm_client=openai_client,
# )

# answer = assistant.rag('I just discovered the course. Can I join now?')
# print(answer)

In [9]:
print(index)

In [56]:
assistant.rag('How do I get a certificate?')
assistant.rag('Can I still join the course after it started?')

'Yes, you can still join after it started. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [ ]:
def llm(prompt):
    #response = openai_client.responses.create(
    response = openai_client.chat.completions.create(
        model='gpt-5.4-mini',
        #input=prompt
        messages=[
            {"role":"system","content": "You are a helpful assistent."},
            {"role":"user","content": prompt},
        ]
    )
    #return response.output_text
    return [ x.message.content for x in response.choices]

In [17]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

['Yes, maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure out the right next step. For example, you could:\n- check the course page for enrollment deadlines\n- contact the instructor or admissions office\n- ask if late registration or an exception is possible\n\nIf you send me the course name or a link, I can help you draft a message asking to join.']


In [18]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''

In [19]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [20]:
answer = llm(prompt)
print(answer)

['Yes, you can still join now. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.']


In [21]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [22]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [28]:
courses_raw[0]

{'course': 'machine-learning-zoomcamp',
 'course_name': 'ML Zoomcamp',
 'path': '/json/machine-learning-zoomcamp.json',
 'questions_count': 472}

In [29]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [30]:
documents[1000]

{'id': '1695e67099',
 'course': 'mlops-zoomcamp',
 'section': 'Module 1: Introduction',
 'question': 'Missing dependencies',
 'answer': 'If some dependencies are missing:\n\n<{IMAGE:image_1}>\n\nInstall the following packages:\n\n- `pandas`\n- `matplotlib`\n- `scikit-learn`\n- `fastparquet`\n- `pyarrow`\n- `seaborn`\n\n```bash\npip install -r requirements.txt\n```\n\nI have seen this error when using `pandas.read_parquet()`. The solution is to install `pyarrow` or `fastparquet` by running the following command in the notebook:\n\n```bash\n!pip install pyarrow\n```\n\n**Note:** If you’re using Conda instead of pip, install `fastparquet` rather than `pyarrow`, as it is much easier to install and it’s functionally identical to `pyarrow` for our needs.'}

In [31]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [ ]:
question = 'I just discovered the course. Can I join now?'

search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

In [34]:
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [35]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [36]:
search_results = search(question)
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [37]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [38]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [39]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [41]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [42]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [43]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)
print(response.output_text)

Yes — you can still join now and start learning.

If you want a certificate, make sure to submit your project while submissions are still open.


In [50]:
response.usage

input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00039900000000000005

In [51]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [52]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [53]:
answer = rag('I just discovered the course. Can I join now?')
print(answer)

Yes, you can still join now. If you want a certificate, make sure you submit your project while submissions are still open.


In [54]:
rag('How do I get a certificate?')

'You can get a certificate only if you finish the course with a live cohort and pass the Capstone project.\n\nA few notes:\n- Self-paced mode does not include certificates.\n- Homework is not required for the certificate.\n- You need to peer-review 3 capstone projects, which is only possible while the course is running.\n- Make sure your official name is set in your course profile if you want it to appear correctly on the certificate.'